<a href="https://colab.research.google.com/github/6pb4wnww4g-beep/Jason/blob/main/Jason_Market_Team_Strength_v2_1_Fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- JASON MARKET TEAM STRENGTH v2.1 FIXED ---
# Fixer desimal-lesing fra Market_xG_All_Fixtures_v1.
#
# Leser:
#   Market_xG_All_Fixtures_v1
#
# Skriver:
#   Market_Decimal_Check_v2_1
#   Market_Team_Strength_v2_1
#   Market_Match_Strength_v2_1

import time
import numpy as np
import pandas as pd
from google.colab import auth
import gspread
from google.auth import default

SPREADSHEET_NAME = "Jason Development"
INPUT_SHEET = "Market_xG_All_Fixtures_v1"

CHECK_OUTPUT_SHEET = "Market_Decimal_Check_v2_1"
TEAM_OUTPUT_SHEET = "Market_Team_Strength_v2_1"
MATCH_OUTPUT_SHEET = "Market_Match_Strength_v2_1"

MIN_SCALE = 10
MAX_SCALE = 95


auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.open(SPREADSHEET_NAME)


def read_sheet_as_text(sheet_name):
    ws = sh.worksheet(sheet_name)
    values = ws.get_all_values()

    if not values or len(values) < 2:
        raise ValueError(f"{sheet_name} er tomt.")

    headers = values[0]
    rows = values[1:]
    return pd.DataFrame(rows, columns=headers)


def update_worksheet(sheet_name, dataframe):
    print(f"Oppdaterer {sheet_name}...")
    df_clean = dataframe.replace([np.inf, -np.inf], np.nan).fillna("")
    try:
        ws = sh.worksheet(sheet_name)
        ws.clear()
    except Exception:
        ws = sh.add_worksheet(title=sheet_name, rows="2000", cols="80")

    ws.resize(
        rows=max(200, len(df_clean) + 20),
        cols=max(25, len(df_clean.columns) + 5)
    )
    ws.update([df_clean.columns.tolist()] + df_clean.values.tolist())
    time.sleep(2)


def parse_sheet_float(v, default=np.nan):
    if v is None:
        return default

    s = str(v).strip()
    if s == "":
        return default

    s = s.replace(" ", "")

    # Norsk desimalkomma
    if "," in s and "." not in s:
        s = s.replace(",", ".")

    # Hvis bÃ¥de komma og punktum finnes: anta komma som tusenskiller
    elif "," in s and "." in s:
        s = s.replace(",", "")

    try:
        return float(s)
    except Exception:
        return default


def scale_10_95(series):
    s = pd.to_numeric(series, errors="coerce")
    mn = s.min()
    mx = s.max()

    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return pd.Series([50.0] * len(series), index=series.index)

    return (MIN_SCALE + (s - mn) / (mx - mn) * (MAX_SCALE - MIN_SCALE)).clip(MIN_SCALE, MAX_SCALE)


df = read_sheet_as_text(INPUT_SHEET)

required = [
    "Kickoff", "Team", "Opponent", "H/A",
    "Team xG", "Opp xG", "CS%", "Win%", "Draw%", "Loss%"
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Mangler kolonner i {INPUT_SHEET}: {missing}")

check_df = df[[
    "Kickoff", "Team", "Opponent", "H/A",
    "Team xG", "Opp xG", "CS%", "Win%", "Draw%", "Loss%"
]].copy()

check_df = check_df.rename(columns={
    "Team xG": "Raw Team xG",
    "Opp xG": "Raw Opp xG",
    "CS%": "Raw CS%",
    "Win%": "Raw Win%",
    "Draw%": "Raw Draw%",
    "Loss%": "Raw Loss%"
})

for col in ["Team xG", "Opp xG", "CS%", "Win%", "Draw%", "Loss%", "Total xG", "Fit Error"]:
    if col in df.columns:
        df[col] = df[col].apply(parse_sheet_float)

df = df.dropna(subset=["Team xG", "Opp xG"]).copy()

if df.empty:
    raise ValueError(f"Ingen gyldige xG-rader funnet i {INPUT_SHEET}")

check_df = check_df.loc[df.index].copy()
check_df["Parsed Team xG"] = df["Team xG"].values
check_df["Parsed Opp xG"] = df["Opp xG"].values
check_df["Check Note"] = ""
check_df.loc[check_df["Team"].astype(str).str.contains("Nottingham", case=False, na=False), "Check Note"] = "Nottingham Team xG skal vÃ¦re ca 1.529, ikke 1529"
check_df.loc[check_df["Opponent"].astype(str).str.contains("Nottingham", case=False, na=False), "Check Note"] = "Opp xG mot Nottingham skal vÃ¦re ca 1.529, ikke 1529"

league_avg_xg = df["Team xG"].mean()

team = (
    df.groupby("Team", as_index=False)
    .agg({
        "Team xG": ["mean", "sum", "count"],
        "Opp xG": ["mean", "sum"],
        "CS%": "mean",
        "Win%": "mean",
        "Draw%": "mean",
        "Loss%": "mean"
    })
)

team.columns = [
    "Team",
    "Avg Team xG",
    "Total Team xG",
    "Matches",
    "Avg Opp xG",
    "Total Opp xG",
    "Avg CS%",
    "Avg Win%",
    "Avg Draw%",
    "Avg Loss%"
]

team["League Avg xG"] = league_avg_xg
team["Team Att Ratio"] = team["Avg Team xG"] / league_avg_xg
team["Team Def Ratio"] = league_avg_xg / team["Avg Opp xG"]

team["Team Att"] = scale_10_95(team["Team Att Ratio"])
team["Team Def"] = scale_10_95(team["Team Def Ratio"])

team["Interpretation"] = team.apply(
    lambda r: f"Att {r['Team Att Ratio']:.2f}x league avg, Def {r['Team Def Ratio']:.2f}x league avg",
    axis=1
)

team_out = team[[
    "Team",
    "Matches",
    "Avg Team xG",
    "Avg Opp xG",
    "League Avg xG",
    "Team Att Ratio",
    "Team Def Ratio",
    "Team Att",
    "Team Def",
    "Avg CS%",
    "Avg Win%",
    "Avg Draw%",
    "Avg Loss%",
    "Total Team xG",
    "Total Opp xG",
    "Interpretation"
]].copy()

team_att_map = dict(zip(team["Team"], team["Team Att"]))
team_def_map = dict(zip(team["Team"], team["Team Def"]))
team_att_ratio_map = dict(zip(team["Team"], team["Team Att Ratio"]))
team_def_ratio_map = dict(zip(team["Team"], team["Team Def Ratio"]))

match = df.copy()

match["Team Att"] = match["Team"].map(team_att_map)
match["Team Def"] = match["Team"].map(team_def_map)
match["Opp Team Att"] = match["Opponent"].map(team_att_map)
match["Opp Team Def"] = match["Opponent"].map(team_def_map)

match["Team Att Ratio"] = match["Team"].map(team_att_ratio_map)
match["Team Def Ratio"] = match["Team"].map(team_def_ratio_map)
match["Opp Team Att Ratio"] = match["Opponent"].map(team_att_ratio_map)
match["Opp Team Def Ratio"] = match["Opponent"].map(team_def_ratio_map)

match["Match Att"] = match["Team Att"] - match["Opp Team Def"]
match["Match Def"] = match["Team Def"] - match["Opp Team Att"]

match["Match Att Ratio Diff"] = match["Team Att Ratio"] - match["Opp Team Def Ratio"]
match["Match Def Ratio Diff"] = match["Team Def Ratio"] - match["Opp Team Att Ratio"]

match_out_cols = [
    "Kickoff",
    "Team",
    "Opponent",
    "H/A",
    "Team Att",
    "Opp Team Def",
    "Match Att",
    "Team xG",
    "Team Def",
    "Opp Team Att",
    "Match Def",
    "Opp xG",
    "CS%",
    "Win%",
    "Draw%",
    "Loss%",
    "Team Att Ratio",
    "Opp Team Def Ratio",
    "Match Att Ratio Diff",
    "Team Def Ratio",
    "Opp Team Att Ratio",
    "Match Def Ratio Diff"
]

match_out = match[match_out_cols].copy()

for out_df in [check_df, team_out, match_out]:
    for col in out_df.columns:
        if pd.api.types.is_numeric_dtype(out_df[col]):
            out_df[col] = out_df[col].round(4)

team_out = team_out.sort_values("Team Att", ascending=False).reset_index(drop=True)
match_out = match_out.sort_values(["Kickoff", "Team"]).reset_index(drop=True)

update_worksheet(CHECK_OUTPUT_SHEET, check_df)
update_worksheet(TEAM_OUTPUT_SHEET, team_out)
update_worksheet(MATCH_OUTPUT_SHEET, match_out)

print("Ferdig.")
print(f"Sjekk fÃ¸rst: {CHECK_OUTPUT_SHEET}")
print(f"Deretter:")
print(f"- {TEAM_OUTPUT_SHEET}")
print(f"- {MATCH_OUTPUT_SHEET}")
print("")
print(f"League Avg xG i grunnlaget: {league_avg_xg:.4f}")
print("Nottingham Forest Avg Team xG skal vÃ¦re ca 1.529 i Team Strength.")
print("Nottingham Team Att Ratio skal vÃ¦re ca 1.04 hvis grunnlaget kun er GW1.")